# Диаризация аудио с NVIDIA NeMo Sortformer

### Установка зависимостей

При работе в google collab -> После первой установки выбрать **Среда выполнения -> Перезапустить сеанс**, затем продолжить с ячейки инициализации ниже, не запуская установку повторно.

In [ ]:
%pip install -q Cython packaging "nemo_toolkit[asr]==2.7.0"

### Инициализация девайса и библиотек

При работе в google collab. После перезапуска начать отсюда.

In [ ]:
import json
import re
import subprocess
from pathlib import Path
from time import perf_counter

import pandas as pd
import psutil
import torch
from google.colab import files
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден. Будет использован CPU.")


### Загрузка аудиофайла

Выбрать один аудиофайл или медиаконтейнер с аудиодорожкой из числа поддерживаемых FFmpeg.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Файл не загружен.")
if len(uploaded) != 1:
    raise ValueError("Загружено несколько файлов, вместо одного.")

audio_name = next(iter(uploaded))
audio_path = Path(audio_name)
allowed_extensions = {
    ".3g2", ".3gp", ".aac", ".ac3", ".aif", ".aifc", ".aiff",
    ".amr", ".ape", ".au", ".avi", ".awb", ".caf", ".dts",
    ".eac3", ".flac", ".flv", ".gsm", ".m2ts", ".m4a", ".m4b",
    ".mka", ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpc",
    ".mpeg", ".mpg", ".mts", ".oga", ".ogg", ".opus", ".ra",
    ".rm", ".snd", ".spx", ".tak", ".ts", ".tta", ".wav",
    ".wave", ".webm", ".wma", ".wv",
}
if audio_path.suffix.lower() not in allowed_extensions:
    raise ValueError(
        f"Неподдерживаемый формат {audio_path.suffix or 'без расширения'}. "
        "Выберите аудиофайл или медиаконтейнер с поддерживаемой аудиодорожкой."
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise ValueError("Загруженный файл отсутствует или пуст.")
print(f"Загружен файл: {audio_name} ({audio_path.stat().st_size / 1024 / 1024:.2f} МБ)")

### Подготовка аудио

In [ ]:
prepared_path = Path("/tmp/prepared_audio.wav")
command = [
    "ffmpeg", "-v", "error", "-y", "-i", str(audio_path),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(prepared_path),
]
try:
    conversion = subprocess.run(command, capture_output=True, text=True, check=False)
except FileNotFoundError as exc:
    raise RuntimeError("FFmpeg not found.") from exc

if conversion.returncode != 0 or not prepared_path.is_file() or prepared_path.stat().st_size <= 44:
    details = (conversion.stderr or "FFmpeg error").strip()[-1000:]
    raise RuntimeError(
        "Не удалось прочитать или преобразовать аудио."
        f"Сообщение FFmpeg: {details}"
    )
print(f"Аудио подготовлено: {prepared_path} (mono, 16 кГц, WAV)")

### Загрузка модели

In [ ]:
from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_ID = "nvidia/diar_sortformer_4spk-v1"
try:
    diar_model = SortformerEncLabelModel.from_pretrained(MODEL_ID)
    diar_model.to(DEVICE)
    diar_model.eval()
except Exception as exc:
    message = str(exc)
    raise RuntimeError(f"Не удалось загрузить модель: {message}") from exc
print(f"Модель загружена на {DEVICE}.")

### Выполнение диаризации

Модель автоматически определяет до четырёх говорящих. Число говорящих вручную не задаётся.

In [ ]:
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
diarization_started_at = perf_counter()

try:
    with torch.inference_mode():
        predicted_segments = diar_model.diarize(
            audio=str(prepared_path),
            batch_size=1,
        )
except (torch.cuda.OutOfMemoryError, MemoryError) as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "Во время диаризации закончилась оперативная или видеопамять. "
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Диаризация завершилась ошибкой: {exc}."
    ) from exc

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    used_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
    memory_measurement_name = "Использование VRAM"
else:
    used_memory_gib = psutil.Process().memory_info().rss / 1024**3
    memory_measurement_name = "Использование RAM"
diarization_elapsed_seconds = perf_counter() - diarization_started_at

raw_segments = predicted_segments
if (
    isinstance(raw_segments, list)
    and len(raw_segments) == 1
    and isinstance(raw_segments[0], list)
):
    raw_segments = raw_segments[0]

def normalize_speaker(value):
    match = re.search(r"(\d+)$", str(value))
    if not match:
        raise ValueError(f"Не удалось определить номер говорящего из {value!r}.")
    return f"SPEAKER_{int(match.group(1)):02d}"

segments = []
for item in raw_segments:
    if isinstance(item, str):
        parts = item.strip().replace(",", " " ).split()
        if len(parts) < 3:
            raise ValueError(f"Неожиданный сегмент Sortformer: {item!r}")
        start, end, speaker = parts[0], parts[1], parts[2]
    elif isinstance(item, (list, tuple)) and len(item) >= 3:
        start, end, speaker = item[0], item[1], item[2]
    else:
        raise ValueError(f"Неожиданный сегмент Sortformer: {item!r}")
    start = float(start)
    end = float(end)
    if end > start:
        segments.append(
            {
                "start": round(start, 3),
                "end": round(end, 3),
                "speaker": normalize_speaker(speaker),
            }
        )
segments.sort(key=lambda item: (item["start"], item["end"], item["speaker"]))
if not segments:
    raise RuntimeError(
        "В записи не обнаружена речь."
    )
print(f"Диаризация завершена. Найдено сегментов: {len(segments)}")


### Просмотр и скачивание временной разметки

Сегменты выводятся в таблице в минутах, полученная временная разметка сохраняется в json(в секундах).

In [ ]:
def format_minutes(seconds):
    minutes, remaining_seconds = divmod(float(seconds), 60)
    return f"{int(minutes):02d}:{remaining_seconds:06.3f}"

table = pd.DataFrame(
    [
        (format_minutes(item["start"]), format_minutes(item["end"]), item["speaker"])
        for item in segments
    ],
    columns=["Начало", "Окончание", "Говорящий"],
)
print(
    f"Время диаризации: {format_minutes(diarization_elapsed_seconds)} "
    f"({diarization_elapsed_seconds:.3f} с)"
)
print(f"{memory_measurement_name}: {used_memory_gib:.3f} GiB")
with pd.option_context("display.max_rows", None):
    display(table)

result = {"audio_file": audio_name, "segments": segments}
result_path = Path(audio_name).with_suffix(".json")
with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Результат сохранён: {result_path.resolve()}")
files.download(str(result_path))
